In [1]:
# Cell 1: Environment Setup

include("scripts/mynordic_transform.jl")
include("scripts/dictionaries_mynordic.jl")
using .MyNordicTransform
using .DictionariesMyNordic
import .MyNordicTransform: om_send
using OMJulia

# --- Configuration ---

# 1. Define the directory containing your models
MODEL_DIR = abspath("MyNordic")

# Path to your local package
MODELS_PKG_PATH = joinpath(MODEL_DIR, "package.mo")

# 2. Define the main model name 
MODEL = "MyNordic.TestCase"

# 3. Define the auxiliary model name
AUX_MODEL = "$(MODEL)_auxiliary"

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH   = "/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 4. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"


"/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

In [2]:
# Cell 2: OpenModelica Bootstrap + Base Model Validation

AUX_PACKAGE = split(MODEL, ".")[1] * "_auxiliary"
AUX_DIR = joinpath(dirname(MODEL_DIR), AUX_PACKAGE)
AUX_ROOT_MODEL = AUX_PACKAGE * "." * split(MODEL, ".")[end] * "_auxiliary"

# Start OMC and load libraries
omc = OMJulia.OMCSession()
om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(omc, "loadModel(Complex)")
om_send(omc, "loadModel(ModelicaServices)")
om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# Load the dynamic model and validate
#om_send(omc, "cd(\"$MODEL_DIR\")")
#om_send(omc, "loadFile(\"$MODEL.mo\")")
om_send(omc, "loadFile(\"$MODELS_PKG_PATH\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($MODEL)", parsed=false)
println(chk)


[ Info: Path to zmq file="/tmp/openmodelica.dyvulgawocfc.port.julia.IwT9zStzSL"


OMC -> loadFile("/home/dyvulgawocfc/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/dyvulgawocfc/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> loadFile("/home/dyvulgawocfc/Notebooks ejemplo/NordicTest/package.mo")
OMC -> clearMessages()
OMC -> checkModel(NordicTest.TestCase)
"Check of NordicTest.TestCase completed successfully.
Class NordicTest.TestCase has 10878 equation(s) and 10878 variable(s).
4364 of these are trivial equation(s)."



In [3]:
# Cell 3: Build Context (Inheritance + Auxiliary Naming + Paths)

parents = om_send(omc, "getInheritedClasses($MODEL)", parsed=false)
println(parents)

chain = get_inheritance_chain(omc, MODEL)
println(chain)

AUX_NAME_MAP = Dict(model => AUX_PACKAGE * "." * split(model, ".")[end] * "_auxiliary" for model in chain)
println(AUX_NAME_MAP)

AUX_PACKAGE_FILE = joinpath(AUX_DIR, "package.mo")
AUX_ORDER_FILE = joinpath(AUX_DIR, "package.order")

println(AUX_DIR)
println(AUX_ROOT_MODEL)


OMC -> getInheritedClasses(NordicTest.TestCase)
{NordicTest.FullDynamicModel, Modelica.Icons.Example}

OMC -> getInheritedClasses(NordicTest.TestCase)
OMC -> getInheritedClasses(NordicTest.FullDynamicModel)
OMC -> getInheritedClasses(NordicTest.NetworkWithAlphaBetaLoads)
OMC -> getInheritedClasses(NordicTest.Network)
["NordicTest.Network", "NordicTest.NetworkWithAlphaBetaLoads", "NordicTest.FullDynamicModel", "NordicTest.TestCase"]
Dict("NordicTest.FullDynamicModel" => "NordicTest_auxiliary.FullDynamicModel_auxiliary", "NordicTest.TestCase" => "NordicTest_auxiliary.TestCase_auxiliary", "NordicTest.Network" => "NordicTest_auxiliary.Network_auxiliary", "NordicTest.NetworkWithAlphaBetaLoads" => "NordicTest_auxiliary.NetworkWithAlphaBetaLoads_auxiliary")
/home/dyvulgawocfc/Notebooks ejemplo/NordicTest_auxiliary
NordicTest_auxiliary.TestCase_auxiliary


In [ ]:
# Cell 4: Component-Specific INIT Model Selection

INIT_MODEL_BY_COMPONENT = Dict{String, String}(
    "g01" => "GeneratorSynchronousExt3W_INIT",
    "g02" => "GeneratorSynchronousExt3W_INIT",
    "g03" => "GeneratorSynchronousExt3W_INIT",
    "g04" => "GeneratorSynchronousExt3W_INIT",
    "g05" => "GeneratorSynchronousExt3W_INIT",
    "g06" => "GeneratorSynchronousExt4W_INIT",
    "g07" => "GeneratorSynchronousExt4W_INIT",
    "g08" => "GeneratorSynchronousExt3W_INIT",
    "g09" => "GeneratorSynchronousExt3W_INIT",
    "g10" => "GeneratorSynchronousExt3W_INIT",
    "g11" => "GeneratorSynchronousExt3W_INIT",
    "g12" => "GeneratorSynchronousExt3W_INIT",
    "g13" => "GeneratorSynchronousExt3W_INIT",
    "g14" => "GeneratorSynchronousExt4W_INIT",
    "g15" => "GeneratorSynchronousExt4W_INIT",
    "g16" => "GeneratorSynchronousExt4W_INIT",
    "g17" => "GeneratorSynchronousExt4W_INIT",
    "g18" => "GeneratorSynchronousExt4W_INIT",
    "g19" => "GeneratorSynchronousExt3W_INIT",
    "g20" => "GeneratorSynchronousExt3W_INIT",
)

println(INIT_MODEL_BY_COMPONENT)


In [ ]:
# Cell 5: Slack Component Selection

SLACK_COMPONENT = "g20"
println(SLACK_COMPONENT)


In [4]:
# Cell 6: Main Auxiliary Build Pipeline

# Main script

# Create an empty auxiliary package in OpenModelica
om_send(omc, "deleteClass($AUX_PACKAGE)")
om_send(omc, "clearMessages()")
om_send(omc, "loadString(\"within ; package $AUX_PACKAGE end $AUX_PACKAGE;\")")

# Build one global blacklist across the full inheritance chain
global_blacklist_names = collect_blacklisted_component_names(omc, chain)

# Loop over the inheritance chain and transform each class
for model in chain
    aux_model = AUX_NAME_MAP[model]
    aux_name = split(aux_model, ".")[end]

    println("Transforming $model -> $aux_model")
    # Copy original class into the auxiliary package
    om_send(omc, "copyClass($model, \"$aux_name\", $AUX_PACKAGE)")

    # Build component dictionary (from the original class)
    components = get_all_components(omc, model)

    # Apply dictionary-driven replacements
    apply_replacements!(omc, model, aux_model, REPLACEMENTS, components, SLACK_COMPONENT)
    # Delete connections to the sources
    delete_connections!(omc, aux_model, components; global_targets = global_blacklist_names)

    # Delete sources
    delete_components!(omc, aux_model, components)

    # Add INIT models (dictionary-driven)
    add_init_models!(omc, model, aux_model, INIT_MODELS, INIT_MODEL_BY_COMPONENT, components, SLACK_COMPONENT)

    # Add extra modifiers necessary for the load-flow
    apply_LF_modifiers!(omc, model, aux_model, INIT_MODELS, components)

    # Add Initial equations
    add_init_equations!(omc, aux_model, components, INIT_MODELS, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)
end

# Create the auxiliary package folder
mkpath(AUX_DIR)

# Write package.mo
open(AUX_PACKAGE_FILE, "w") do io
    print(io, "within ;\npackage $AUX_PACKAGE\nend $AUX_PACKAGE;\n")
end

# Write package.order
open(AUX_ORDER_FILE, "w") do io
    for model in chain
        println(io, split(AUX_NAME_MAP[model], ".")[end])
    end
end

# Save each transformed auxiliary class
for model in chain
    aux_model = AUX_NAME_MAP[model]
    aux_name = split(aux_model, ".")[end]
    aux_file = joinpath(AUX_DIR, aux_name * ".mo")

    # Get the current transformed class text from OpenModelica
    txt = String(om_send(omc, "listFile($aux_model)"))

    # Rewrite extends(...) so the auxiliary classes inherit from the auxiliary parents
    txt = rewrite_aux_extends(txt, AUX_NAME_MAP)

    # Save the auxiliary class
    open(aux_file, "w") do io
        print(io, txt)
        endswith(txt, "\n") || print(io, "\n")
    end

    # Apply the same text-level cleanup as in the single-file workflow
    patch_aux_equations!(aux_file)

    # Remove TestCase omegaCOI/generatorSynchronous references after generator swaps
    if aux_name == "TestCase_auxiliary"
        patch_testcase_omega_refs!(aux_file)
    end
end

# Re-load generated auxiliary package from disk and validate
om_send(omc, "deleteClass($AUX_PACKAGE)")
om_send(omc, "loadFile(\"$AUX_PACKAGE_FILE\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($AUX_ROOT_MODEL)", parsed=false)
println(chk)


OMC -> deleteClass(NordicTest_auxiliary)
OMC -> clearMessages()
OMC -> loadString("within ; package NordicTest_auxiliary end NordicTest_auxiliary;")
Transforming NordicTest.Network -> NordicTest_auxiliary.Network_auxiliary
OMC -> copyClass(NordicTest.Network, "Network_auxiliary", NordicTest_auxiliary)
OMC -> loadClassContentString("initial equation", NordicTest_auxiliary.Network_auxiliary)
Transforming NordicTest.NetworkWithAlphaBetaLoads -> NordicTest_auxiliary.NetworkWithAlphaBetaLoads_auxiliary
OMC -> copyClass(NordicTest.NetworkWithAlphaBetaLoads, "NetworkWithAlphaBetaLoads_auxiliary", NordicTest_auxiliary)
OMC -> updateComponent(load_22, Dynawo.Electrical.Loads.LoadPQ, NordicTest_auxiliary.NetworkWithAlphaBetaLoads_auxiliary, modification = $Code((u0Pu = u0Pu_load_22, i0Pu = i0Pu_load_22, s0Pu = s0Pu_load_22)))
OMC -> updateComponent(load_71, Dynawo.Electrical.Loads.LoadPQ, NordicTest_auxiliary.NetworkWithAlphaBetaLoads_auxiliary, modification = $Code((u0Pu = u0Pu_load_71, i0Pu = 